In [0]:
yellowtaxiDF = spark.read.csv("/Volumes/azuredatabricks2239/default/yellowtaxi/yellow_tripdata_2016-01.csv", header=True, inferSchema=True)

In [0]:
# print schema of yellowTaxi Table
print(yellowtaxiDF.schema)

StructType([StructField('VendorID', IntegerType(), True), StructField('tpep_pickup_datetime', TimestampType(), True), StructField('tpep_dropoff_datetime', TimestampType(), True), StructField('passenger_count', IntegerType(), True), StructField('trip_distance', DoubleType(), True), StructField('pickup_longitude', DoubleType(), True), StructField('pickup_latitude', DoubleType(), True), StructField('RatecodeID', IntegerType(), True), StructField('store_and_fwd_flag', StringType(), True), StructField('dropoff_longitude', DoubleType(), True), StructField('dropoff_latitude', DoubleType(), True), StructField('payment_type', IntegerType(), True), StructField('fare_amount', DoubleType(), True), StructField('extra', DoubleType(), True), StructField('mta_tax', DoubleType(), True), StructField('tip_amount', DoubleType(), True), StructField('tolls_amount', DoubleType(), True), StructField('improvement_surcharge', DoubleType(), True), StructField('total_amount', DoubleType(), True)])


In [0]:
yellowtaxiDF.coalesce(1).write.mode("overwrite").format("parquet").save("/Volumes/azuredatabricks2239/default/yellowparquettaxi/")

### Bronze Table

In [0]:
%sql
CREATE LIVE TABLE YellowTaxi_Bronze(
  VendorID integer,
  tpep_pickup_datetime timestamp,
  tpep_dropoff_datetime timestamp,
  passenger_count integer,
  trip_distance double,
  pickup_longitude double,
  pickup_latitude double,
  RatecodeID integer,
  store_and_fwd_flag string,
  dropoff_longitude double,
  dropoff_latitude double,
  payment_type integer,
  fare_amount double,
  extra double,
  mta_tax double,
  tip_amount double,
  tolls_amount double,
  improvement_surcharge double,
  total_amount double,
  FileName String,
  CreatedOn Timestamp
) USING DELTA PARTITIONED BY (VendorID)
AS 
SELECT * , INPUT_FILE_NAME() as FileName, CURRENT_TIMESTAMP() as CreatedOn
FROM parquet.`/Volumes/azuredatabricks2239/default/yellowparquettaxi/`

Name,Type
VendorID,int
tpep_pickup_datetime,timestamp
tpep_dropoff_datetime,timestamp
passenger_count,int
trip_distance,double
pickup_longitude,double
pickup_latitude,double
RatecodeID,int
store_and_fwd_flag,string
dropoff_longitude,double


### Silver Table

In [0]:
%sql
-- Generated as Keyword by Databricks SQL to indicate that PickUpYear ccolumn is auto-computed column, its value will be automatically calculated based on the expression year(tpep_pickup_datetime)
create live table YellowTaxi_Silver(
  VendorID integer,
  tpep_pickup_datetime timestamp,
  tpep_dropoff_datetime timestamp,
  passenger_count integer,
  trip_distance double,
  pickup_longitude double,
  pickup_latitude double,
  RatecodeID integer,
  payment_type integer,
  fare_amount double,
  total_amount double,
  PickUpYear INT generated always as (year (tpep_pickup_datetime)),
  PickUpMonth INT generated always as (month (tpep_pickup_datetime)),
  PickUpDay INT generated always as (day (tpep_pickup_datetime)),
  CreatedOn timestamp,
  constraint ValidTotalAmount expect(total_amount is not null and total_amount>0) on violation drop row,
  constraint ValidDistance expect(trip_distance is not null and trip_distance>0) on violation drop row,
  constraint ValidPassenger expect(passenger_count>0) on violation drop row,
  constraint ValidVendorID expect(VendorID is not null and VendorID>0) on violation fail update
) using delta partitioned by (passenger_count)
as select vendorID, tpep_pickup_datetime, tpep_dropoff_datetime, passenger_count, trip_distance, pickup_longitude, pickup_latitude, RatecodeID, payment_type, fare_amount, current_timestamp() as CreatedOn
from live.YellowTaxi_Bronze

Name,Type
VendorID,int
tpep_pickup_datetime,timestamp
tpep_dropoff_datetime,timestamp
passenger_count,int
trip_distance,double
pickup_longitude,double
pickup_latitude,double
RatecodeID,int
payment_type,int
fare_amount,double


In [0]:
%sql
CREATE LIVE TABLE YellowTaxi_Gold(
  AS
  SELECT RatecodeID, passenger_count,
  count(*) AS TotalRide,
  SUM(trip_distance) AS TotalDistance),
  SUM(total_amount) AS TotalAmount